# Trigger wave thresholds

In [1]:
import immunowave as iw
import numpy as np
import jax
import jax.numpy as jnp
import diffrax as dx

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from multiprocessing import Pool
import multiprocessing as mp
from functools import partial

import pickle

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

from hill_function_utils import (
    tissue_response,
    single_cell_response,
    compute_wave_threshold,
    compute_cell_threshold,
)

import warnings

warnings.simplefilter("ignore")

from immunowave_paper_utils import style_axes, colors, fontsize, linewidth, rc_params
from diffrax import SaveAt

In [2]:
%matplotlib qt

In [3]:
linewidth = 4
fontsize = 24
markersize = 24
markeredgewidth = 4
rc_params["axes.linewidth"] = linewidth
rc_params["font.size"] = fontsize
mpl.rcParams.update(rc_params)
mpl.rcParams["pdf.fonttype"] = 42

In [4]:
"""ok, so multiprocessing with jax only works if you use a
non-default start method within the multiprocessing library.
Here I use forkserver. An annoying corollary of forkserver
(and also spawn) is that in jupyter notebooks, functions
called within a Pool() need to be imported and can't be
defined in another cell. So I moved the functions to a
python file in this same directory."""

mp.set_start_method("forkserver")

## Model definition

Here the model is defined with slightly different notation from the main text:

$\partial_t A = D\partial^2_x A + \frac{A^n}{K_D^n + A^n} - \gamma A + \eta B\delta(x)$

In [5]:
def Bc_wave_theory(n, KD, gamma, D, a):
    return (
        2
        * (1 - 2 / (n + 1)) ** (1 / 2)
        * (KD * gamma) ** (n / (n - 1))
        * np.sqrt(D / gamma)
        / a
    )

In [6]:
class State4(iw.State):
    A: iw.ScalarField
    B: iw.ScalarField


class HillModel(iw.Model):
    KD: float
    n: int
    η: float
    ξ: float
    λ: float
    μ: float
    D: float = 1.0
    gamma: float = 1.0

    @jax.jit
    def __call__(self, t, state, args=None):
        # unpack field variables
        A, B = state.A, state.B
        # unpack parameters
        KD, n, η, ξ, λ, μ, D, gamma = (
            self.KD,
            self.n,
            self.η,
            self.ξ,
            self.λ,
            self.μ,
            self.D,
            self.gamma,
        )
        # define PDE
        An = A.binop(n, jnp.power)
        tmp = An + KD**n
        tmp = tmp.binop(-1, jnp.power)
        hill_term = An * tmp
        dAdt = D * A.laplacian(bc="neumann") + hill_term - gamma * A + η * B
        dBdt = ξ * B.laplacian(bc="neumann") + λ * B * (1 - B) - μ * A * B
        return State4(dAdt, dBdt)


# @jax.jit
def response(B0, KD, t_max, hill_coefficient=2):
    model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0)
    state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        B=iw.ScalarField(
            shape,
            lb,
            h,
            fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1),
        ),
    )
    solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        t=jnp.array([t_max]),
        **kwargs,
    )
    return np.sum(solution.ys.A.values[-1] * solution.ys.A.h)
    # return solution.evaluate(t_final).A.integral() / L


def find_B0(final_mean_A, B0s):
    this_id = np.where(np.diff(final_mean_A) == np.max(np.diff(final_mean_A)))[0][0]
    B0c = 0.5 * (B0s[this_id] + B0s[this_id + 1])
    uncertainty = 0.5 * (B0s[this_id + 1] - B0s[this_id])
    return B0c, uncertainty


## Compute critical stimulus strength as a function of various parameters

### Vary $K_D$

In [7]:
KDs = np.logspace(-2, np.log10(0.15), 20)
B0s = np.logspace(-4, 0, 40)
t_max = 100
L = 100.0
n = 2000  # 200
D = 1.0
gamma = 1.0
hill_coefficient = 4
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5
scale = 0.1

"""wave"""
Bc_wave = np.zeros(len(KDs))
Bc_wave_uncertainties = np.zeros(len(KDs))
for i, KD in enumerate(KDs):
    print(f"{i + 1} of {len(KDs)}")
    tissue_theory = (
        2
        * (1 - (2 / (hill_coefficient + 1))) ** 0.5
        * KD ** (hill_coefficient / (hill_coefficient - 1))
    )
    B0s = np.linspace(0.5 * tissue_theory, 2 * tissue_theory, 40)
    Bc_wave[i], Bc_wave_uncertainties[i] = compute_wave_threshold(
        B0s,
        KD=KDs[i],
        t_max=t_max,
        hill_coefficient=hill_coefficient,
        D=D,
        gamma=gamma,
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        n_iters=n_iters,
        scale=scale,
        **kwargs,
    )

1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KeyboardInterrupt: 

In [16]:
KD_sweep_dict = {
    "KDs": KDs,
    "Bc_wave": Bc_wave,
    "Bc_wave_uncertainties": Bc_wave_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v3_scale0pt1.pkl",
    "wb",
) as f:
    pickle.dump(KD_sweep_dict, f)

In [8]:
"""plot"""
# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v3_scale0pt1.pkl",
    "rb",
) as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict["KDs"]
Bc_wave = KD_sweep_dict["Bc_wave"]

plt.figure(figsize=(8, 7))
plt.plot(
    KDs,
    Bc_wave,
    marker="o",
    markersize=18,
    markerfacecolor="none",
    markeredgewidth=4,
    linestyle="none",
    markeredgecolor=colors["wave"],
    label="wave numerics",
)
tissue_theory = (
    2
    * (1 - (2 / (hill_coefficient + 1))) ** 0.5
    * KDs ** (hill_coefficient / (hill_coefficient - 1))
)
plt.plot(
    KDs,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)

plt.legend(fontsize=18)
plt.xscale("log")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel(
    "concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize
)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()

### Vary hill coefficients

In [20]:
"""vary hill coefficients"""

KD = 0.01  
B0s = np.logspace(-5, -2, 40)
t_max = 100
L = 100.0
n = 2000
hill_coefficients = np.linspace(2, 5, 20)  
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
scale = 0.1
n_iters = 5

"""tissue"""
print("tissue")
Bc_wave = np.zeros(len(hill_coefficients))
Bc_wave_uncertainties = np.zeros(len(hill_coefficients))
for i in range(len(hill_coefficients)):
    print(f"{i + 1} of {len(hill_coefficients)}")
    tissue_theory = (
        2
        * (1 - (2 / (hill_coefficients[i] + 1))) ** 0.5
        * KD ** (hill_coefficients[i] / (hill_coefficients[i] - 1))
    )
    B0s = np.linspace(0.5 * tissue_theory, 2 * tissue_theory, 40)
    Bc_wave[i], Bc_wave_uncertainties[i] = compute_wave_threshold(
        B0s,
        KD=KD,
        t_max=t_max,
        hill_coefficient=hill_coefficients[i],
        D=D,
        gamma=gamma,
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        scale=scale,
        n_iters=n_iters,
        **kwargs,
    )


hill_coeff_sweep_dict = {
    "hill_coefficients": hill_coefficients,
    "Bc_wave": Bc_wave,
    "Bc_wave_uncertainties": Bc_wave_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/hill_coefficient_sweep_scale0pt1.pkl",
    "wb",
) as f:
    pickle.dump(hill_coeff_sweep_dict, f)

tissue
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KeyboardInterrupt: 

In [25]:
hill_coeff_sweep_dict = {
    "hill_coefficients": hill_coefficients,
    "Bc_wave": Bc_wave,
    "Bc_cell": Bc_cell,
    "Bc_wave_uncertainties": Bc_wave_uncertainties,
    "Bc_cell_uncertainties": Bc_cell_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/initial-theory-numerics-comparison/hill_coefficient_sweep.pkl",
    "wb",
) as f:
    pickle.dump(hill_coeff_sweep_dict, f)

In [9]:
"""plot"""

ns = np.linspace(2, 5, 100)

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/hill_coefficient_sweep_scale0pt1.pkl",
    "rb",
) as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients = hill_coeff_sweep_dict["hill_coefficients"]
Bc_wave = hill_coeff_sweep_dict["Bc_wave"]

plt.figure(figsize=(8, 7))
plt.plot(
    hill_coefficients,
    Bc_wave,
    marker="o",
    markersize=18,
    markerfacecolor="none",
    markeredgewidth=4,
    linestyle="none",
    markeredgecolor=colors["wave"],
    label="wave numerics",
)
tissue_theory = 2 * (1 - (2 / (ns + 1))) ** 0.5 * KD ** (ns / (ns - 1))
plt.plot(
    ns,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)


plt.legend(fontsize=18)
plt.xscale("linear")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel("hill coefficient, $n$", fontsize=fontsize)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()

### Vary decay rate

In [39]:
"""vary decay rate. now time in minutes. length in microns"""

B0s = np.logspace(-6, -3, 40)
KD = 0.01  
t_maxs = np.linspace(2000, 100, 20)
L = 100.0
n = 2000
hill_coefficient = 3
gammas = np.linspace(0.02, 0.2, 20)
D = 1.0
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5
scale = 0.1


"""tissue"""
print("tissue")
Bc_wave = np.zeros(len(gammas))
Bc_wave_uncertainties = np.zeros(len(gammas))
for i in range(len(gammas)):
    print(f"{i + 1} of {len(gammas)}")
    Bc_wave[i], Bc_wave_uncertainties[i] = compute_wave_threshold(
        B0s,
        KD=KD,
        t_max=t_maxs[i],
        hill_coefficient=hill_coefficient,
        D=D,
        gamma=gammas[i],
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        scale=scale,
        n_iters=n_iters,
        **kwargs,
    )


gamma_sweep_dict = {
    "gammas": gammas,
    "Bc_wave": Bc_wave,
    "Bc_wave_uncertainties": Bc_wave_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/gamma_sweep_scale0pt1_longer-time.pkl",
    "wb",
) as f:
    pickle.dump(gamma_sweep_dict, f)

tissue
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3 of 20


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

19 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

20 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [47]:
gamma_sweep_dict = {
    "gammas": gammas,
    "Bc_wave": Bc_wave,
    "Bc_cell": Bc_cell,
    "Bc_wave_uncertainties": Bc_wave_uncertainties,
    "Bc_cell_uncertainties": Bc_cell_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/initial-theory-numerics-comparison/gamma_sweep.pkl",
    "wb",
) as f:
    pickle.dump(gamma_sweep_dict, f)

In [12]:
"""plot"""
# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/gamma_sweep_scale0pt1_longer-time.pkl",
    "rb",
) as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_sweep_dict["gammas"]
Bc_wave = gamma_sweep_dict["Bc_wave"]
gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)
hill_coefficient = 3

plt.figure(figsize=(8, 7))
plt.plot(
    gammas,
    Bc_wave,
    marker="o",
    markersize=18,
    markerfacecolor="none",
    markeredgewidth=4,
    linestyle="none",
    markeredgecolor=colors["wave"],
    label="wave numerics",
)
tissue_theory = (
    2
    * (1 - (2 / (hill_coefficient + 1))) ** 0.5
    * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1))
    * np.sqrt(D / gamma_theory)
)
plt.plot(
    gamma_theory,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)

plt.legend(fontsize=18)
plt.xscale("linear")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel(r"loss rate, $\gamma$ (1/min)", fontsize=24)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()

## Plot everything including schematics

In [144]:
plt.close("all")

In [7]:
f, axs = plt.subplots(2, 3, figsize=(18.97, 10.79))

### Solution branches schematic

In [5]:
def derivative_func(A, hill_coefficient, KD):
    return A**hill_coefficient / (KD**hill_coefficient + A**hill_coefficient) - A
    # return A * (A - 0.4) * (1 - A)

In [8]:
KD = 0.4
hill_coefficient = 4

ax = axs[0, 0]
ax.clear()

A = np.linspace(0, 1.1, 10000)
V = np.cumsum(derivative_func(A[:-1], hill_coefficient, KD) * np.diff(A)[0])
E = 0
dAdx = (2 * (E - V)) ** 0.5
max_dAdx = np.nanmax(dAdx)

ax.plot(A[:-1], dAdx, "-", linewidth=linewidth, color=(0.5, 0.5, 0.5))
ax.plot(A[:-1], -dAdx, "-", linewidth=linewidth, color=(0.5, 0.5, 0.5))

# find E = V(u3), where u3 is the high fixed point
pol = [-1, 1, 0, 0, -(KD**hill_coefficient), 0]
roots = np.roots(pol)
big_root = np.max(roots)
E = V[np.where(np.abs(A - big_root) == np.min(np.abs(A - big_root)))[0][0]]
dAdx = (2 * (E - V)) ** 0.5
ax.plot(A[:-1], dAdx, "-", linewidth=linewidth, color=colors["wave"])
ax.plot(A[:-1], -dAdx, "-", linewidth=linewidth, color=colors["wave"])


ax.plot(
    0,
    0,
    "s",
    markersize=markersize,
    markeredgecolor="r",
    markeredgewidth=markeredgewidth,
    markerfacecolor="none",
)
ax.plot(
    roots[1],
    0,
    "o",
    markersize=markersize,
    markeredgecolor="r",
    markeredgewidth=markeredgewidth,
    markerfacecolor="none",
)
ax.plot(
    roots[0],
    0,
    "s",
    markersize=markersize,
    markeredgecolor="r",
    markeredgewidth=markeredgewidth,
    markerfacecolor="none",
)

xline = np.linspace(-0.1, 1.1, 5)
ax.plot(xline, max_dAdx * np.ones_like(xline), "--", linewidth=linewidth, color="k")
ax.plot(xline, -max_dAdx * np.ones_like(xline), "--", linewidth=linewidth, color="k")

ax.set_xlabel("$u$ (a.u.)", fontsize=fontsize)
ax.set_ylabel("$\partial_x u$ (a.u.)", fontsize=fontsize)
ax.set_title("steady state\nphase plane", fontsize=fontsize)
ax.set_xlim([-0.1, 1.1])
# ax.set_ylim([-1.5 * max_dAdx, 1.5 * max_dAdx])
ax = style_axes(ax, fontsize=fontsize)

### Wave threshold schematic

In [148]:
KD = 0.4
t_max = 300  # 45
L = 3 * 100.0
n = 2000
hill_coefficient = 4
gamma = 1.0
D = 1.0
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)

"""below thresh"""
B0 = 1.22 * Bc_wave_theory(hill_coefficient, KD, gamma, D, a=1)
model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, gamma=gamma)
state = State4(
    A=iw.ScalarField(shape, lb, h, 0),
    B=iw.ScalarField(
        shape,
        lb,
        h,
        fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=0.5),
    ),
)

solution = iw.solve(
    model,
    state,
    t0=0,
    t1=t_max,
    saveat=SaveAt(ts=jnp.linspace(0, t_max, 8)),
    **kwargs,
)


A = solution.ys.A.values
x = np.linspace(0, L, n)

ax = axs[0, 1]
ax.clear()
ax.plot(x, A[-1], "-", linewidth=linewidth, color=(0.5, 0.5, 0.5))

ax.set_xlabel("space, $x$ (a.u.)", fontsize=fontsize)
ax.set_ylabel("$u$ (a.u.)", fontsize=fontsize)
ax.set_title(f"local response\n$I={B0:.3f}$", fontsize=fontsize)
ax.ticklabel_format(scilimits=(-3, 6))
ax = style_axes(ax, fontsize=fontsize)


"""above thresh"""
B0 = 1.23 * Bc_wave_theory(hill_coefficient, KD, gamma, D, a=1)
model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, gamma=gamma)
state = State4(
    A=iw.ScalarField(shape, lb, h, 0),
    B=iw.ScalarField(
        shape,
        lb,
        h,
        fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=0.5),
    ),
)

solution = iw.solve(
    model,
    state,
    t0=0,
    t1=t_max,
    saveat=SaveAt(ts=jnp.linspace(0, t_max, t_max)),
    **kwargs,
)


A = solution.ys.A.values
x = np.linspace(0, L, n)
n_plot_ts = 6  # 8
plot_ts = np.linspace(32, len(A), n_plot_ts, dtype="int")
plot_reds = np.linspace(0.5, colors["wave"][0], 8)
plot_greens = np.linspace(0.5, colors["wave"][1], 8)
plot_blues = np.linspace(0.5, colors["wave"][2], 8)

ax = axs[0, 2]
ax.clear()
for i in range(n_plot_ts):
    ax.plot(
        x,
        A[plot_ts[i]],
        "-",
        linewidth=linewidth,
        color=(plot_reds[i], plot_greens[i], plot_blues[i]),
    )

ax.set_xlabel("space, $x$ (a.u.)", fontsize=fontsize)
ax.set_ylabel("$u$ (a.u.)", fontsize=fontsize)
ax.set_title(f"trigger wave\n$I={B0:.3f}$", fontsize=fontsize)
# ax.set_ylim([0, 1])
ax = style_axes(ax, fontsize=fontsize)

### comparison with numerics --- KD

In [168]:
"""plot"""

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v3_scale0pt1.pkl",
    "rb",
) as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict["KDs"]
Bc_wave = KD_sweep_dict["Bc_wave"]
hill_coefficient = 4

ax = axs[1, 0]
ax.clear()
KDs_theory = np.logspace(np.log10(np.min(KDs)), np.log10(np.max(KDs)), 1000)
tissue_theory = (
    2
    * (1 - (2 / (hill_coefficient + 1))) ** 0.5
    * KDs_theory ** (hill_coefficient / (hill_coefficient - 1))
)
ax.plot(
    KDs_theory,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="analytic",
)


ax.plot(
    KDs,
    Bc_wave,
    marker="o",
    markersize=markersize,
    markerfacecolor="none",
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=colors["wave"],
    label="numerics",
)

ax.legend(fontsize=fontsize)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(
    "concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize
)
ax.set_ylim([1e-3, 5e-1])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='concentration scale \nof positive feedback, $K_D$ (a.u.)', ylabel='$I^{wave}_{c}$ (a.u.)'>

### comparison with numerics--- vary hill coeff

In [164]:
"""plot"""

ns = np.linspace(2, 5, 100)
KD = 0.01

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/hill_coefficient_sweep_scale0pt1.pkl",
    "rb",
) as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients = hill_coeff_sweep_dict["hill_coefficients"]
Bc_wave = hill_coeff_sweep_dict["Bc_wave"]

ax = axs[1, 1]
ax.clear()

tissue_theory = 2 * (1 - (2 / (ns + 1))) ** 0.5 * KD ** (ns / (ns - 1))
ax.plot(
    ns,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)

ax.plot(
    hill_coefficients,
    Bc_wave,
    marker="o",
    markersize=markersize,
    markerfacecolor="none",
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=colors["wave"],
    label="wave numerics",
)


ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel("hill coefficient, $n$", fontsize=fontsize)
ax.set_ylim([5e-5, 1e-2])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='hill coefficient, $n$', ylabel='$I^{wave}_{c}$ (a.u.)'>

### comparision with numerics---vary gamma

In [161]:
"""plot"""

KD = 0.01

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/gamma_sweep_scale0pt1_longer-time.pkl",
    "rb",
) as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_sweep_dict["gammas"]
Bc_wave = gamma_sweep_dict["Bc_wave"]

gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)
hill_coefficient = 3

ax = axs[1, 2]
ax.clear()

tissue_theory = (
    2
    * (1 - (2 / (hill_coefficient + 1))) ** 0.5
    * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1))
    * np.sqrt(D / gamma_theory)
)
ax.plot(
    gamma_theory,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)

ax.plot(
    gammas,
    Bc_wave,
    marker="o",
    markersize=markersize,
    markerfacecolor="none",
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=colors["wave"],
    label="wave numerics",
)

ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(r"decay rate, $\gamma$ (1/min)", fontsize=fontsize)
ax.set_ylim([1e-5, 1e-3])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='decay rate, $\\gamma$ (1/min)', ylabel='$I^{wave}_{c}$ (a.u.)'>

In [158]:
plt.gcf().tight_layout()

In [173]:
plt.savefig(
    r"/home/brandon/Documents/Code/immunowave/plots/2025-04-15_numerics_comparision_fig-wave_v2.pdf"
)